 SDXL_Research
Caption, train, compare and generate SDXL LoRAs on the Colab free T4.

1. **Runtime > Change runtime type > T4 GPU**
2. Add your Hugging Face token in **Secrets** (key icon) as `HF_TOKEN` (needed for private repos)
3. Run **Setup** once per session; re-runs skip finished steps.

In [ ]:
#@title Setup: repo + venv (idempotent)
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/kopeng-mi/SDXL_Research.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO = Path("/content/SDXL_Research")

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "pull", "-q", "--ff-only"], cwd=REPO, check=True)
    subprocess.run(["git", "submodule", "update", "-q", "--init", "--recursive"], cwd=REPO, check=True)

# The kernel only orchestrates (stdlib); GPU work runs in venv subprocesses,
# so this kernel never holds VRAM.
sys.path.insert(0, str(REPO / "src"))
from sdxlr import nb
r = subprocess.run([sys.executable, "-m", "sdxlr.bootstrap"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                   env={**os.environ, "PYTHONPATH": str(REPO / "src")})
print(r.stdout[-3000:])
assert r.returncode == 0, "setup failed (see above)"
nb.hf_token()

# Dataset
Puts images (and any existing `.txt` captions) into one folder.
`test` downloads 5 public dog photos to try the pipeline end to end.

In [ ]:
#@title Get dataset
SOURCE = "test"  #@param ["test", "hf", "drive"]
DATASET_DIR = "/content/dataset"  #@param {type:"string"}
#@markdown **hf**: private/public repo + zip filename
HF_REPO = ""  #@param {type:"string"}
HF_FILE = "dataset.zip"  #@param {type:"string"}
#@markdown **drive**: folder or .zip in your Drive
DRIVE_PATH = "/content/drive/MyDrive/datasets/my_images"  #@param {type:"string"}

args = ["-m", "sdxlr.data", "--dest", DATASET_DIR]
if SOURCE == "test":
    args += ["--source", "test"]
elif SOURCE == "hf":
    args += ["--source", "hf", "--repo", HF_REPO, "--file", HF_FILE]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    args += ["--source", "dir", "--path", DRIVE_PATH]
nb.run(args)

# Caption
Writes one `.txt` per image. Captions describe clothing, pose, background and
lighting, and skip face and body, so the LoRA learns those from the pixels.
Check the first few with `PREVIEW = 3`, then set `PREVIEW = 0` for all.

In [ ]:
#@title Caption (Qwen3-VL / JoyCaption)
MODEL = "qwen"  #@param ["qwen", "qwen-2b", "joy"]
TRIGGER = "ohwx_person"  #@param {type:"string"}
CLASS_WORD = "woman"  #@param {type:"string"}
PROMPT = ""  #@param {type:"string"}
#@markdown blank PROMPT = identity preset; `{cls}` in a custom prompt = CLASS_WORD
PREVIEW = 3  #@param {type:"integer"}
OVERWRITE = False  #@param {type:"boolean"}

args = ["-m", "sdxlr.caption", "--images", DATASET_DIR, "--model", MODEL,
        "--trigger", TRIGGER, "--class-word", CLASS_WORD, "--limit", PREVIEW]
if PROMPT: args += ["--prompt", PROMPT]
if OVERWRITE: args += ["--overwrite"]
nb.run(args)

# Train
Defaults: `configs/train/sdxl_person.yaml` (proven T4 settings).
With `HF_REPO` set, LoRAs + resume state upload to that **private** repo each
epoch; tick `RESUME` after a disconnect to continue.

In [ ]:
#@title Train SDXL LoRA
RUN_NAME = "test_run"  #@param {type:"string"}
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0/sd_xl_base_1.0.safetensors"  #@param {type:"string"}
#@markdown org/repo/file on HF, an https URL, or a local path
HF_REPO = ""  #@param {type:"string"}
RESUME = False  #@param {type:"boolean"}
EPOCHS = 15  #@param {type:"integer"}
REPEATS = 6  #@param {type:"integer"}
BATCH = 3  #@param {type:"integer"}
LORA_TYPE = "LoRA"  #@param ["LoRA", "LoCon"]
DIM = 24  #@param {type:"integer"}
ALPHA = 16  #@param {type:"integer"}
UNET_LR = 2e-4  #@param {type:"number"}
TE_LR = 2e-5  #@param {type:"number"}
EXTRA = ""  #@param {type:"string"}
#@markdown EXTRA: more overrides as `key=value key=value` (see the yaml)

args = ["-m", "sdxlr.train", "--name", RUN_NAME, "--images", DATASET_DIR, "--model", BASE_MODEL,
        "--set", f"epochs={EPOCHS}", f"repeats={REPEATS}", f"batch_size={BATCH}",
        f"lora_type={LORA_TYPE}", f"network_dim={DIM}", f"network_alpha={ALPHA}",
        f"unet_lr={UNET_LR}", f"text_encoder_lr={TE_LR}", *EXTRA.split()]
if HF_REPO: args += ["--hf-repo", HF_REPO]
if RESUME: args += ["--resume"]
nb.run(args)

# Train (Z-Image)
LoRA for Z-Image via musubi-tuner, trained on the Z-Image **base** model and
used on **Turbo** in the app. Defaults: `configs/train/zimage_person.yaml`.
On the T4 this trains at 512px (1024px runs out of memory).

In [ ]:
#@title Train Z-Image LoRA
RUN_NAME = "z_test_run"  #@param {type:"string"}
RESOLUTION = 512  #@param [512, 768] {type:"raw"}
EPOCHS = 10  #@param {type:"integer"}
REPEATS = 4  #@param {type:"integer"}
DIM = 16  #@param {type:"integer"}
LR = 1e-4  #@param {type:"number"}
EXTRA = ""  #@param {type:"string"}
#@markdown EXTRA: more overrides as `key=value key=value` (see the yaml)

nb.run(["-m", "sdxlr.train_zimage", "--name", RUN_NAME, "--images", DATASET_DIR,
        "--set", f"resolution={RESOLUTION}", f"epochs={EPOCHS}", f"repeats={REPEATS}",
        f"network_dim={DIM}", f"network_alpha={DIM}", f"lr={LR}", *EXTRA.split()])

# Models and LoRAs
Downloads to `/content/work`. LoRAs you train in this session are picked up
automatically. Comma-separate several sources; `.zip` files of LoRAs are unpacked.

In [ ]:
#@title Fetch base model + LoRAs
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0/sd_xl_base_1.0.safetensors"  #@param {type:"string"}
#@markdown org/repo/file on HF, an https URL, or a local path
LORAS = ""  #@param {type:"string"}
#@markdown e.g. `me/loras/set1.zip, me/loras/face_b.safetensors`

nb.run(["-m", "sdxlr.fetch", "model", BASE_MODEL])
if LORAS.strip():
    nb.run(["-m", "sdxlr.fetch", "loras", *[s.strip() for s in LORAS.split(",") if s.strip()]])

# Compare
Rows = LoRAs (or LoRA mixes), columns = strengths, same seed everywhere.
Re-running continues where it stopped; change any setting to start a new grid.

In [ ]:
#@title LoRA comparison grid
GRID_NAME = "grid1"  #@param {type:"string"}
MODEL_TYPE = "SDXL"  #@param ["SDXL", "Z-Image"]
Z_MODEL = "turbo Q8"  #@param ["turbo Q8", "turbo Q6", "turbo Q5", "turbo Q4"] {allow-input: true}
Z_TEXT_ENCODER = "official"  #@param ["official", "abliterated"] {allow-input: true}
#@markdown Z-Image: use STEPS 9, CFG 0; prompt weights like (word:1.2) are SDXL-only
LORA_GLOB = "*.safetensors"  #@param {type:"string"}
COMBOS = ""  #@param {type:"string"}
#@markdown COMBOS (optional) replaces LORA_GLOB: `face_a.safetensors:0.44+face_b.safetensors:0.44, other.safetensors:0.8`. Strengths then multiply the mix.
STRENGTHS = "0.78 0.82 0.88"  #@param {type:"string"}
PROMPT = "(ohwx_person:1.2), woman, highly detailed face, from thigh up, white background"  #@param {type:"string"}
NEGATIVE = "(worst quality:1.3), (low quality:1.2), lowres, bad anatomy, bad hands, watermark, blur, cartoon, anime, drawing"  #@param {type:"string"}
STEPS = 25  #@param {type:"integer"}
CFG = 5  #@param {type:"number"}
SAMPLER = "dpmpp_2m_karras"  #@param ["dpmpp_2m_karras", "dpmpp_2m_sde_karras", "dpmpp_sde_karras", "euler_a", "euler"]
CLIP_SKIP = 2  #@param {type:"integer"}
WIDTH = 896  #@param {type:"integer"}
HEIGHT = 1152  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}

model_args = (["--model", Z_MODEL, "--zimage", "--te", Z_TEXT_ENCODER] if MODEL_TYPE == "Z-Image"
              else ["--model", BASE_MODEL])
args = ["-m", "sdxlr.compare", "--name", GRID_NAME, *model_args, "--strengths", *STRENGTHS.split(),
        "--prompt", PROMPT, "--negative", NEGATIVE, "--steps", STEPS, "--cfg", CFG, "--sampler", SAMPLER,
        "--clip-skip", CLIP_SKIP, "--width", WIDTH, "--height", HEIGHT, "--seed", SEED]
combos = [c.strip() for c in COMBOS.split(",") if c.strip()]
args += ["--combos", *combos] if combos else ["--loras", LORA_GLOB]
nb.run(args)
from IPython.display import Image as _Img, display
display(_Img(filename=f"/content/work/compare/{GRID_NAME}/grid.png"))

# Generate (UI)
Opens the app; the cell keeps running while you use it. Stop the cell to
close the app and free all VRAM (do that before training).

In [ ]:
#@title Start the app
SHARE_LINK = False  #@param {type:"boolean"}
#@markdown off: Colab's private proxy link (only you). on: public gradio.live link as a fallback.
LOAD_AT_START = "SDXL"  #@param ["SDXL", "Z-Image", "none"]
#@markdown SDXL uses BASE_MODEL from the fetch cell. Z-Image: pick the model and text encoder (or paste any org/repo/file.gguf / org/repo).
Z_MODEL = "turbo Q8"  #@param ["turbo Q8", "turbo Q6", "turbo Q5", "turbo Q4"] {allow-input: true}
Z_TEXT_ENCODER = "official"  #@param ["official", "abliterated"] {allow-input: true}
Z_ENCODER_PRECISION = "nf4"  #@param ["nf4", "int8", "fp16"]

try:
    from google.colab.output import eval_js
    print("App link (opens when the app says it is running):", eval_js("google.colab.kernel.proxyPort(7860)"))
except Exception:
    pass
args = ["-m", "sdxlr.app", "--port", 7860]
if SHARE_LINK: args += ["--share"]
if LOAD_AT_START == "SDXL":
    args += ["--model", BASE_MODEL]
elif LOAD_AT_START == "Z-Image":
    args += ["--zimage", Z_MODEL, "--te", Z_TEXT_ENCODER, "--te-quant", Z_ENCODER_PRECISION]
nb.run(args)